# Example 1 — Basic LTE planning

Plan connections between towers, sectors, and clients using the planning service. The supplied scenario defines the sites, antennas, interfaces, and metrics.

Before starting, install SDK 0.1.18 or later with the `notebook` extra (pandas and Matplotlib). Open this notebook from the downloaded `notebooks` folder, with the kernel working in that folder. The project at `../projects/example1_LTE_planning_basic` contains `scenario.toml`, `nodes.csv`, and the reusable `features.json` and saved `result.json` artifacts when they exist. Both planning and features services must be available. Run each cell in order.

## 1. Connect to the planning service

In [ ]:
import pandas as pd
from cisei_planning_sdk import PlanningClient, PlanningProject, plot_network

client = PlanningClient(
    "http://network-service:8080",
    workspace="..",
    timeout=900,
)


## 2. Load the example scenario

The SDK reads the project’s scenario and nodes, then imports `features.json` if it exists. Missing features are extracted during computation.

In [ ]:
project = PlanningProject("../projects/example1_LTE_planning_basic")
scenario = client.open_project(project, planner="cell")

scenario.summary


## 3. Build possible connections

Sector direction and beamwidth determine which tower-to-client connections are considered.

In [ ]:
candidates = scenario.build_candidate_edges()

pd.DataFrame(candidates["sectors"])


## 4. Calculate link metrics

The service retrieves geographic features and evaluates the metric configured in the scenario. Only cache misses require the geo service at `http://planning-service:8080`. After computation, the SDK saves the raw features in the project’s `features.json` for future sessions.

In [ ]:
metrics = scenario.compute_metrics()

pd.DataFrame(metrics["metrics"])


## 5. Plan the network

The solver selects connections using the calculated link costs.

In [ ]:
solution = scenario.solve()

pd.DataFrame(solution["planned_edges"])


## 6. Examine service quality

The rank threshold defines the maximum acceptable accumulated cost. Here we use a threshold of 2.

In [ ]:
evaluation = scenario.evaluate(rank_threshold=2)

evaluation["quality_summary"]


Which clients are served, and by which cell?

In [ ]:
pd.DataFrame(evaluation["cell"]["client_service"])


How many clients does each cell serve?

In [ ]:
pd.DataFrame(evaluation["cell"]["cell_load"])


## 7. Save the result

Save the network and its quality evaluation in this example’s project as `result.json`. The export also includes the candidate connections needed for the graph views.

In [ ]:
scenario.save_result(rank_threshold=2)


## 8. Load the saved result

The following views use only the local file. They do not require the original server session.


In [ ]:
saved_result = scenario.load_result()


### Candidate network

All connections considered during planning. Each marker represents a site; interfaces at the same site are grouped together.


In [ ]:
plot_network(saved_result, kind="candidate", save_path=project.path / "candidate_network.html");


### Planned network

Selected connections, with connected sites, served clients, and unserved clients distinguished by color. These are projected network views, not street maps.


In [ ]:
plot_network(saved_result, kind="final", save_path=project.path / "solution_network.html");
